# Module 2 — Steering + MMLU capability check (Llama-3.1-8B-Instruct)

**v2 §3 capability gate, Llama parallel track.** Sweep α ∈ {0.025, 0.05, 0.075, 0.1} per emotion and measure MMLU under steering. Report the largest α whose drop from the Llama baseline is ≤ 1pt.

**Prerequisite:** M1 Llama must have dropped vectors into `outputs/m1_vectors/llama_L{layer}/{emotion}.npy` (or to the HF Hub equivalent path; `load_emotion_vector(model_key='llama')` will pull from HF if missing locally).

**Note:** the drop tolerance is relative to *Llama's own* baseline MMLU, not Gemma's. Llama-3.1-8B baseline MMLU is ~68%; Gemma-2-9B baseline is ~72%. The 1pt gate logic is unchanged.

**Cost:** ~3 GPU-hr on A100 (1 baseline + 16 steered runs × ~1140 MMLU prompts each).

## Cell 1 — env setup (Colab + SageMaker + local)

On SageMaker, expects `HF_TOKEN` already in env and that you opened this notebook from inside the cloned `Algoverse/` directory.

In [ ]:
import os
import sys
import subprocess

# detect runtime
IS_COLAB = 'google.colab' in sys.modules
IS_SAGEMAKER = os.path.exists('/home/ec2-user/SageMaker') or 'SageMaker' in os.environ.get('PWD', '')
print(f'runtime: colab={IS_COLAB}, sagemaker={IS_SAGEMAKER}')

# secrets — Colab uses userdata; SageMaker/local expects them already in env
if IS_COLAB:
	from google.colab import userdata
	os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
	assert 'HF_TOKEN' in os.environ, (
		'HF_TOKEN not set. Before launching Jupyter, run in the terminal:\n'
		'  export HF_TOKEN=hf_...\n'
		'and accept the model license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct'
	)

# repo: Colab clones fresh each session; SageMaker/local expects you started in the repo
if IS_COLAB:
	subprocess.run('git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)', shell=True, check=True)
	os.chdir('Algoverse')
else:
	assert os.path.isdir('src') and os.path.isfile('config.yaml'), (
		f'expected to be inside the Algoverse repo root, got cwd={os.getcwd()}. '
		'On SageMaker: `cd ~/SageMaker/Algoverse` and re-open this notebook from there.'
	)

# pip install is idempotent — fine to re-run on each session
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'cwd: {os.getcwd()}')


## Cell 1.4 — pre-flight: GPU, disk, dep check

In [ ]:
# pre-flight: GPU, disk, deps. Cheap — run before the model load to catch problems early.
import subprocess
import torch
import transformers

print('=== GPU ===')
try:
	print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], text=True))
except FileNotFoundError:
	print('nvidia-smi not found — CPU-only environment, model load will OOM')

print('=== Disk (cwd) ===')
print(subprocess.check_output(['df', '-h', '.'], text=True))

print('=== Deps ===')
print(f'torch={torch.__version__}, cuda={torch.version.cuda}, transformers={transformers.__version__}')
print(f'cuda available: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}')

# Llama-3.1-8B bf16 ≈ 16 GB weights; need ~20 GB free with activations + KV cache headroom
if torch.cuda.is_available():
	free_gb = torch.cuda.mem_get_info()[0] / 1e9
	print(f'free VRAM: {free_gb:.1f} GB')
	if free_gb < 20:
		print('warning: <20 GB free VRAM. Llama-8B bf16 may OOM during generation.')


## Cell 1.45 — confirm HF Write access to the artifact repo

Same fail-fast probe as M1 Llama. M2 produces ~16 MMLU prediction CSVs that need to land in the artifact repo at the end — verifying Write access now means a 403 won't orphan them.

In [ ]:
# whose HF token is active, and does it have Write access to the artifact repo?
# fail fast — discovering the 403 after 8 GPU-hours is a budget-killer. This probe
# uploads a tiny placeholder then immediately deletes it, so nothing ends up visible
# in the repo's file tree. (Commits remain in history — that's how HF works.)
import os
import tempfile
from huggingface_hub import whoami, upload_file, delete_file

from src.lib.config import load_config
cfg = load_config()
REPO_ID = cfg['paths']['hf_artifact_repo']

who = whoami()
HF_USER = who['name']
# expose to later cells so HF upload commit messages can attribute the run
os.environ['HF_USER'] = HF_USER
print(f'HF identity: {HF_USER}')
print(f'target repo: {REPO_ID}')

# write-access probe: upload then delete
_probe_in_repo = '.write_access_probe'
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as _f:
	_f.write('write-access probe (auto-deleted)\n')
	_probe_path = _f.name
try:
	upload_file(
		path_or_fileobj=_probe_path,
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: write-access check',
	)
except Exception as e:
	raise RuntimeError(
		f'\nHF Write to {REPO_ID} FAILED for user {HF_USER}:\n  {e}\n\n'
		f'Fix: the repo owner needs to add you as a Write collaborator at\n'
		f'  https://huggingface.co/datasets/{REPO_ID}/settings\n'
		f'Owner navigates to Settings -> Collaborators -> Add user -> {HF_USER} -> Write.\n'
		f'Stop the notebook here; do NOT burn GPU-hours until this is fixed.'
	)

# clean up so the probe doesn't show in the file tree of a public repo
try:
	delete_file(
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: cleanup',
	)
	print(f'OK — Write access confirmed; probe file deleted from repo tree.')
except Exception as e:
	print(f'warning: probe upload OK but delete failed ({e}); the placeholder file may be visible at HEAD until manually removed')


## Cell 1.5 — pull M1 Llama vectors from HF

In [ ]:
import os, subprocess

from huggingface_hub import snapshot_download
from src.lib.config import load_config, layer_suffix

cfg = load_config()
lsuf = layer_suffix(cfg, 'llama')
snapshot_download(
        repo_id=cfg['paths']['hf_artifact_repo'],
        repo_type='dataset',
        allow_patterns=[f'm1_vectors/{lsuf}/*'],
        local_dir='outputs',
)
print(subprocess.check_output(['ls', f'outputs/m1_vectors/{lsuf}/'], text=True))


## Cell 2 — load Llama and verify M1 vectors are on disk

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import numpy as np

from src.lib.config import load_config
from src.lib.model_load import load_llama
from src.steering import load_emotion_vector

cfg = load_config()
emotions = cfg['extraction']['emotions']
layer = cfg['models']['llama']['extraction_layer']
alphas = cfg['steering']['alpha_sweep']
drop_tol = cfg['steering']['mmlu_drop_tolerance']

vectors = {e: load_emotion_vector(e, model_key='llama') for e in emotions}
for e, v in vectors.items():
    print(f'{e}: shape={v.shape}, ||v||={np.linalg.norm(v):.4f}')

model, tokenizer = load_llama()
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}, steering layer={layer}')


## Cell 3 — estimate mean residual norm at the steering layer

α is a fraction of `||h||` at the target layer (Anthropic convention). Calibrated on the shared neutral corpus. One pass through 20 passages.

In [ ]:
from src.steering import estimate_residual_norm

data_dir = Path(cfg['paths']['data_dir'])
neutral_texts = [p.read_text(encoding='utf-8') for p in sorted((data_dir / 'stories' / 'neutral').glob('*.txt'))]
print(f'calibrating on {len(neutral_texts)} neutral passages')

norm_scale = estimate_residual_norm(
    model, tokenizer, layer=layer,
    calibration_texts=neutral_texts,
    token_skip=cfg['extraction']['token_skip'],
)
print(f'mean ||h|| at layer {layer} = {norm_scale:.2f}')
print(f'α=0.025 → steering magnitude ≈ {0.025 * norm_scale:.2f}')
print(f'α=0.10  → steering magnitude ≈ {0.10 * norm_scale:.2f}')


## Cell 4 — unsteered MMLU baseline

Stratified 20 prompts × 57 subjects = 1140 prompts, log-prob scoring over (A, B, C, D). ~10 min on A100.

In [ ]:
from src.mmlu_eval import run_mmlu

baseline = run_mmlu(model, tokenizer)
print(f'Llama baseline MMLU: {baseline.accuracy:.4f} ({baseline.n_correct}/{baseline.n_total})')


## Cell 5 — sweep α × emotions

16 runs total. Each ~10 min → ~3 hr wall clock. Hooks registered before MMLU and removed after.

In [ ]:
from src.steering import make_steering_hook_factory
import pandas as pd

outputs_dir = Path(cfg['paths']['outputs_dir']) / 'm2' / lsuf
outputs_dir.mkdir(parents=True, exist_ok=True)

rows = []
for emotion in emotions:
    for alpha in alphas:
        print(f'\n=== {emotion} @ α={alpha} ===')
        hook_factory = make_steering_hook_factory(
            vector=vectors[emotion],
            layer=layer,
            alpha=alpha,
            norm_scale=norm_scale,
        )
        result = run_mmlu(model, tokenizer, pre_forward_hook=hook_factory)
        drop = baseline.accuracy - result.accuracy
        rows.append({
            'emotion': emotion,
            'alpha': alpha,
            'accuracy': result.accuracy,
            'n_correct': result.n_correct,
            'n_total': result.n_total,
            'drop_from_baseline': drop,
            'within_tolerance': drop <= drop_tol,
        })
        result.per_row.to_csv(outputs_dir / f'predictions_{emotion}_a{alpha}.csv', index=False)
        print(f'acc={result.accuracy:.4f}, drop={drop:+.4f}, within {drop_tol:.2f} tol: {drop <= drop_tol}')

df = pd.DataFrame(rows)
df.to_csv(outputs_dir / 'mmlu_by_alpha_emotion.csv', index=False)
df


## Cell 6 — apply capability gate, report per-emotion max α

For each emotion, the result is the largest α from the sweep with `drop ≤ drop_tol`. If no α satisfies the gate, that emotion has no usable steering strength under this protocol on Llama — note it; do not pad.

In [ ]:
lines = ['Module 2 — Steering + MMLU capability check (Llama-3.1-8B-Instruct)', '=' * 60, '']
lines.append(f'baseline MMLU accuracy: {baseline.accuracy:.4f} ({baseline.n_correct}/{baseline.n_total})')
lines.append(f'drop tolerance: {drop_tol:.2f}')
lines.append(f'norm scale ||h||@L{layer}: {norm_scale:.2f}')
lines.append('')
lines.append('per-emotion max usable α:')

for emotion in emotions:
    sub = df[(df['emotion'] == emotion) & (df['within_tolerance'])].sort_values('alpha', ascending=False)
    if len(sub):
        winner = sub.iloc[0]
        lines.append(f'  {emotion:13s}  α={winner["alpha"]:.4f}  acc={winner["accuracy"]:.4f}  drop={winner["drop_from_baseline"]:+.4f}')
    else:
        lines.append(f'  {emotion:13s}  NO usable α in sweep — all α exceeded drop tolerance')

lines.append('')
lines.append('full grid:')
lines.append(df.to_string(index=False))

decision = '\n'.join(lines)
print(decision)
(outputs_dir / 'decision.txt').write_text(decision, encoding='utf-8')


## Outputs

- `outputs/m2/llama_L{layer}/mmlu_by_alpha_emotion.csv`
- `outputs/m2/llama_L{layer}/predictions_{emotion}_a{alpha}.csv`
- `outputs/m2/llama_L{layer}/decision.txt`

## Push to HF Hub

In [ ]:
import os
from huggingface_hub import HfApi, create_repo

repo_id = cfg['paths']['hf_artifact_repo']
create_repo(repo_id, repo_type='dataset', private=True, exist_ok=True)
hf_user = os.environ.get('HF_USER', 'unknown')
HfApi().upload_folder(
    folder_path=f'outputs/m2/{lsuf}',
    repo_id=repo_id,
    repo_type='dataset',
    path_in_repo=f'm2_results/{lsuf}',
    commit_message=f'M2 Llama-3.1-8B-Instruct: MMLU sweep across {{emotion,alpha}} (by HF user {hf_user})',
)


## Next

The per-emotion winning α here is the α used in M3 Llama (steered FaithEval). If desperation has no usable α, the Llama steering hypothesis fails the capability gate and M3 Llama should be reconsidered.